In [1]:
def global_edit_distance(x, y):
    """
    Computes standard edit distance between two sequences from start to end.
    Forces global alignment; looks at the bottom-right corner.
    """
    # Create distance matrix
    D = []
    for i in range(len(x)+1):
        D.append([0]*(len(y)+1))
    # Initialize first row and column of matrix
    for i in range(len(x)+1):
        D[i][0] = i
    for i in range(len(y)+1):
        D[0][i] = i ## This forces a global alignment check from position 0
    # Fill in the rest of the matrix
    for i in range(1, len(x)+1):
        for j in range(1, len(y)+1):
            distHor = D[i][j-1] + 1
            distVer = D[i-1][j] + 1
            if x[i-1] == y[j-1]:
                distDiag = D[i-1][j-1]
            else:
                distDiag = D[i-1][j-1] + 1
            D[i][j] = min(distHor, distVer, distDiag)
    # Edit distance is the value in the bottom right corner of the matrix
    return D[-1][-1]


In [10]:
def approximate_substring_match(x, y):
    """
    Finds the best approximate match of a pattern P within a larger text T.
    Allows the match to start anywhere without penalty; looks for the minimum in the bottom row.
    """
    # Create distance matrix
    D = []
    for i in range(len(x)+1):
        D.append([0]*(len(y)+1))
    # Initialize first row and column of matrix
    for i in range(len(x)+1):
        D[i][0] = i
    for j in range(len(y)+1):
        D[0][j] = 0  # This allows the match to start anywhere in the text
    # Fill in the rest of the matrix
    for i in range(1, len(x)+1):
        for j in range(1, len(y)+1):
            distHor = D[i][j-1] + 1
            distVer = D[i-1][j] + 1
            if x[i-1] == y[j-1]:
                distDiag = D[i-1][j-1]
            else:
                distDiag = D[i-1][j-1] + 1
            D[i][j] = min(distHor, distVer, distDiag)
  
    # To this:
    return min(D[-1])  # The minimum value in the bottom row


In [12]:
def read_fasta(filename):
    """Reads a FASTA file and returns the sequence as a single string."""
    sequence = []
    with open(filename, 'r') as f:
        for line in f:
            # Skip the header line
            if line.startswith('>'):
                continue
            # Remove whitespace/newlines and add to list
            sequence.append(line.strip())
    return ''.join(sequence)

In [5]:
# 1. Define the test inputs
pattern_example = 'GCGTATGC'
text_example = 'TATTGGCTATACGGTT'

# 2. Run the global alignment function
global_distance = global_edit_distance(pattern_example, text_example)

# 3. Print the results clearly
print("=" * 50)
print("RUNNING: GLOBAL EDIT DISTANCE")
print("=" * 50)
print(f"Pattern (X): '{pattern_example}'")
print(f"Text (Y):    '{text_example}'")
print("-" * 50)
print(f"Total Edit Distance (End-to-End): {global_distance}")
print("=" * 50)

RUNNING: GLOBAL EDIT DISTANCE
Pattern (X): 'GCGTATGC'
Text (Y):    'TATTGGCTATACGGTT'
--------------------------------------------------
Total Edit Distance (End-to-End): 11


In [11]:
# 1. Define the test inputs (using the hint from your instructions)
pattern_example = 'GCGTATGC'
text_example = 'TATTGGCTATACGGTT'

# 2. Run the approximate substring match function
substring_distance = approximate_substring_match(pattern_example, text_example)

# 3. Print the results clearly
print("=" * 50)
print("RUNNING: APPROXIMATE SUBSTRING MATCH")
print("=" * 50)
print(f"Pattern (P): '{pattern_example}'")
print(f"Text (T):    '{text_example}'")
print("-" * 50)
print(f"Minimum Edit Distance found in Text: {substring_distance}")
print("Hint Check:  (Should be 2) -> Match Success!")
print("=" * 50)

RUNNING: APPROXIMATE SUBSTRING MATCH
Pattern (P): 'GCGTATGC'
Text (T):    'TATTGGCTATACGGTT'
--------------------------------------------------
Minimum Edit Distance found in Text: 2
Hint Check:  (Should be 2) -> Match Success!


In [15]:
# 1. Define your inputs
p = 'GATTTACCAGATTGAG'  # Change this to whatever pattern

# ---> ENTER YOUR FILE LOADING FUNCTION HERE <---
t = read_fasta('chr1.GRCh38.excerpt.fasta')

# 2. Run the approximate substring match function
print(f"Scanning text sequence for pattern '{p}'... (This might take a moment)")
min_edits = approximate_substring_match(p, t)

# 3. Print the execution results clearly
print("=" * 60)
print("APPROXIMATE SUBSTRING MATCH EXECUTION")
print("=" * 60)
print(f"Pattern (P):             {p}")
print(f"Total Text Length (T):   {len(t)} bases")
print("-" * 60)
print(f"Minimum Edits Required:  {min_edits}")
print("=" * 60)

Scanning text sequence for pattern 'GATTTACCAGATTGAG'... (This might take a moment)
APPROXIMATE SUBSTRING MATCH EXECUTION
Pattern (P):             GATTTACCAGATTGAG
Total Text Length (T):   800000 bases
------------------------------------------------------------
Minimum Edits Required:  2


In [16]:
def overlap(a, b, min_length=3):
    """ Return length of longest suffix of 'a' matching
        a prefix of 'b' that is at least 'min_length'
        characters long.  If no such overlap exists,
        return 0. """
    start = 0  # start all the way at the left
    while True:
        start = a.find(b[:min_length], start)  # look for b's prefix in a
        if start == -1:  # no more occurrences to right
            return 0
        # found occurrence; check for full suffix/prefix match
        if b.startswith(a[start:]):
            return len(a)-start
        start += 1  # move just past previous match

In [23]:

filename = "ERR266411_1.for_asm.fastq"


# --- Helper 2: Parse FASTQ ---
def read_fastq(filename):
    """Extracts only the DNA sequence strings from a FASTQ file."""
    sequences = []
    with open(filename, 'r') as f:
        while True:
            f.readline() # Line 1: Identifier (@)
            seq = f.readline().rstrip() # Line 2: Sequence
            f.readline() # Line 3: +
            f.readline() # Line 4: Quality Scores
            if len(seq) == 0:
                break
            sequences.append(seq)
    return sequences

# --- Helper 3: Exact Overlap Finder ---
def overlap(a, b, min_length=3):
    start = 0
    while True:
        start = a.find(b[:min_length], start)
        if start == -1:
            return 0
        if b.startswith(a[start:]):
            return len(a) - start
        start += 1

# --- Main Driver Script ---
# 1. Load data
reads = read_fastq(filename)
k = 30

# 2. Step 1: Build the k-mer index dictionary
print("\nBuilding the k-mer index mapping...")
kmer_dict = {}
for read in reads:
    for i in range(len(read) - k + 1):
        kmer = read[i:i+k]
        if kmer not in kmer_dict:
            kmer_dict[kmer] = set()
        kmer_dict[kmer].add(read)

# 3. Step 2: Traverse reads and calculate edges using index lookups
print("Scanning for valid overlaps...")
edge_count = 0
distinct_pairs = set()

for read_a in reads:
    # Target the length-k suffix of read_a
    suffix_kmer = read_a[-k:]
    
    # Check if this k-mer exists in any other reads
    if suffix_kmer in kmer_dict:
        # Only evaluate reads that contain this exact suffix k-mer
        candidate_reads = kmer_dict[suffix_kmer]
        
        for read_b in candidate_reads:
            # Skip overlapping a read with itself
            if read_a == read_b:
                continue
                
            # Verify full overlap match
            olen = overlap(read_a, read_b, min_length=k)
            if olen >= k:
                edge_count += 1
                distinct_pairs.add((read_a, read_b))
#  Output results
print("=" * 50)
print(f"Total Number of Overlap Graph Edges: {edge_count}")
print(f"Number of Distinct Pairs:            {len(distinct_pairs)}")
print("=" * 50)



Building the k-mer index mapping...
Scanning for valid overlaps...
Total Number of Overlap Graph Edges: 904746
Number of Distinct Pairs:            904746


In [18]:
# Assuming 'reads', 'kmer_dict', and 'overlap' are already loaded from your previous cell

print("Scanning for nodes with outgoing edges...")
outgoing_nodes = set()  # Tracks unique reads that have at least one outgoing edge

for read_a in reads:
    # Target the length-k suffix of read_a
    suffix_kmer = read_a[-k:]
    
    # Check if this k-mer exists in our lookup index
    if suffix_kmer in kmer_dict:
        candidate_reads = kmer_dict[suffix_kmer]
        
        for read_b in candidate_reads:
            # Skip overlapping a read with itself
            if read_a == read_b:
                continue
                
            # Verify if the full overlap meets the threshold
            olen = overlap(read_a, read_b, min_length=k)
            if olen >= k:
                # read_a has a valid outgoing edge to read_b!
                outgoing_nodes.add(read_a) 
                
                # Optimization: Once we know read_a has at least one 
                # outgoing edge, we don't need to check its other pairs.
                break 

# Output the node calculation result
print("=" * 50)
print(f"Number of nodes with >= 1 outgoing edge: {len(outgoing_nodes)}")
print("=" * 50)

Scanning for nodes with outgoing edges...
Number of nodes with >= 1 outgoing edge: 7161
